# Forecast error plots
Load an available residual checkpoint and forecast one event with the shared rollout and diagnostic functions.
The old tide dataset/checkpoint are not included in this repository; these defaults use the available drag-target example.

In [ ]:
using Pkg
repository = @__DIR__
while !isfile(joinpath(repository, "src", "Surge.jl"))
    parent = dirname(repository)
    parent == repository && error("Open this notebook from within the repository.")
    global repository = parent
end
Pkg.activate(joinpath(repository, "Starting_networks"))
include(joinpath(repository, "src", "Surge.jl"))
import .Surge
using Plots, Statistics
include(joinpath(repository, "src", "experiment_diagnostics.jl"))

In [ ]:
dataset_file = joinpath(repository, "Data", "drag_target_10.jld2")
pretrained_file = joinpath(repository, "Trained_models", "drag_source_pretrained.bson")
event_index = 1

In [ ]:
dataset = Surge.load_burgers_ensemble(basename(dataset_file);
    data_path=dirname(dataset_file), relative_times=true)
model = Surge.load_model(pretrained_file)
event = dataset.solution[event_index]

In [ ]:
forecast = Surge.forecast_event(Surge.effective_model(model), event, dataset.params)
water_level_indices = collect(eachindex(dataset.grid.h))
series = ExperimentDiagnostics.error_series(forecast.prediction, forecast.target,
    first(event).h, dataset.params, water_level_indices, water_level_indices)
mean_error = (h=mean(row.mse_h for row in series), u=mean(row.mse_u for row in series))
println("Mean rollout MSE: ", mean_error)

In [ ]:
for (metric, label) in ((:mse_h, "Water level"), (:mse_u, "Velocity"))
    display(plot(dataset.times ./ 3600, [getproperty(row, metric) for row in series];
        xlabel="Forecast [hours]", ylabel="MSE", title="$label rollout error", label=label))
end